# 00 — Walkthrough: the modeling workspace

**What this notebook is.** The front door to cfdb's predictive modeling. It loads the training data, checks it is the file we think it is, shows how the games are split for training and testing, demonstrates the guard that stops a model seeing the answer, and prints **the market's own record** — the bar every model we build has to clear.

**What it is not.** There is no model here yet. The first models arrive in round cfdb-wtc-R-2420, and they will be scored with exactly the functions used below.

**Licence.** The data comes from the CFB Model Training Pack, which is licensed for personal use only. This notebook reads it from disk at run time; the pack itself is never committed. This notebook is committed **without its outputs** for the same reason — run it to see the numbers.

## 1. Setup

**What:** make the `modeling` package importable from here, and give DataFrames a readable style.

**Why:** the notebook lives two folders below the repository root, so Python cannot find `modeling/` on its own. Walking up to the folder that contains it works wherever the checkout is. The table style sets both the background *and* the text colour on every cell, so tables read the same in a light theme, a dark theme and on paper.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "modeling" / "__init__.py").exists())
sys.path.insert(0, str(ROOT))

from modeling.data import load_pack
from modeling.splits import split
from modeling.leakage import LeakageError, assert_no_leakage, pack_feature_columns
from modeling.evaluate import evaluate

# One dict of colours, formatted into the CSS below — a change is one hex edit.
PALETTE = {"border": "#8e99a6", "grid": "#c3cbd5", "ink": "#1a1a1a", "paper": "#ffffff",
           "head_bg": "#2f4156", "head_ink": "#ffffff", "index_bg": "#eceff4",
           "stripe": "#f6f8fa", "hover": "#fff6d5"}
display(HTML('''<style>
table.dataframe {{ border-collapse: collapse !important; border: 1px solid {border} !important;
  background-color: {paper} !important; color: {ink} !important; font-size: 12px !important; }}
table.dataframe td, table.dataframe th {{ border: 1px solid {grid} !important; padding: 4px 8px !important;
  text-align: left !important; }}
table.dataframe thead th {{ background-color: {head_bg} !important; color: {head_ink} !important;
  font-weight: 600 !important; border-color: {head_bg} !important; position: sticky; top: 0; z-index: 2; }}
table.dataframe tbody th {{ background-color: {index_bg} !important; color: {ink} !important; font-weight: 600 !important; }}
table.dataframe tbody td {{ background-color: {paper} !important; color: {ink} !important; }}
table.dataframe tbody tr:nth-child(even) td {{ background-color: {stripe} !important; }}
table.dataframe tbody tr:hover td {{ background-color: {hover} !important; }}
</style>'''.format(**PALETTE)))
print("repository root:", ROOT)

## 2. Load the pack and check its contract

**What:** read `training_data.csv` and run three checks on every row: the row count is 5,133; `margin` equals `away_points − home_points`; and season 2020 is absent.

**Why:** a model trained on the wrong file does not crash — it produces numbers, and they look like any other numbers. The contract turns "wrong file" into an error message instead. The margin check also pins the **sign convention**: a *negative* margin means the **home** team won, and a *negative* spread means the **home** team was favoured. We never flip it.

In [ ]:
frame = load_pack()          # raises ContractError if any check fails
print(f"{len(frame):,} games × {frame.shape[1]} columns — contract passed")
frame.groupby(["season", "season_type"]).size().unstack(fill_value=0)

## 3. Split by season, never by shuffling

**What:** train on 2016–2023, choose between models on 2024, and test on 2025 — regular season only.

**Why:** a random shuffle would let the model learn from a Week 9 game and then be tested on the Week 8 game before it — grading it on the past with knowledge of the future. Splitting by season keeps every test game strictly after everything the model learned from. The 2025 test set is used **once** per set of candidates, so it stays an honest exam rather than something we tune against.

In [ ]:
parts = split(frame)
pd.DataFrame({name: {"games": len(p), "seasons": ", ".join(map(str, sorted(p.season.unique())))}
              for name, p in parts.items()}).T

## 4. The leakage guard

**What:** every list of model inputs passes through `assert_no_leakage`, which refuses the final score, anything computed from it, and the closing spread.

**Why:** *leakage* is a model seeing, while it learns, something it could not know before kickoff. It does not break anything — it makes the model look brilliant in testing and useless on a real Saturday. The spread is refused for a different reason: it is the benchmark we are graded against, and a model fed the line just learns to repeat it.

Below: the pack's pre-game features pass, and adding `spread` is refused by name.

In [ ]:
features = pack_feature_columns(frame.columns)
print(f"{len(features)} pre-game features offered to a model, e.g. {features[:4]}")

try:
    assert_no_leakage(features + ["spread"])
except LeakageError as refused:
    print("\nAdding the closing spread:", refused)

## 5. The market's own record — the bar to clear

**What:** score the closing line itself on the 2024 (validate) and 2025 (test) games. The line's predicted margin is simply the spread.

**Why:** a model's error means nothing on its own — "off by 12 points" is excellent in a season of blowouts and poor in a season of close games. Every number a model produces will be printed beside these, on the same games.

**How to read it:**
- **margin MAE** — in a typical game, how many points the line missed the final margin by.
- **straight-up %** — how often the favourite won.
- **ATS %** — the market has no against-the-spread record (it *is* the spread), so that row shows the **52.4% break-even**: at standard −110 odds you must win 52.4% of bets just to break even. That is the number a model has to beat to make money. Pushes — a final margin exactly equal to the spread — are excluded and counted.
- **total MAE** — blank for the market: the pack carries no closing total.

In [ ]:
pd.concat({f"{int(p.season.iloc[0])} ({name})": evaluate(p)[["market", "games", "note"]]
           for name, p in [("validate", parts["validate"]), ("test", parts["test"])]}).round(3)

## 6. What comes next

cfdb-wtc-R-2420 trains the first models on the `train` split, chooses between them on `validate`, and scores the chosen ones **once** on `test`, with `evaluate(test, pred_margin=..., pred_total=...)` filling in the model column beside the market's.